In [1]:
import pandas as pd
import glob

In [2]:
csv_files = glob.glob("*.csv")

def analyze_dataset(data):
    # Defining the groups based on row ranges
    # Note: data.iloc[start:stop] handles the slicing
    groups = {
        "combinational_basic": data.iloc[0:80],
        "sequential_basic":    data.iloc[80:160],
        "fsm":                 data.iloc[160:190],
        "industry":            data.iloc[-10:] # Gets the last 10 rows
    }

    results = {}

    for name, df in groups.items():
        if not df.empty:
            # Calculate % where iverilog_output is "OK"
            success_pct = (df['iverilog_output'] == 'OK').mean() * 100

            # Calculate average of time(s)
            avg_time = df['time(s)'].mean()

            avg_prompt_tkns =  df['prompt_tkns'].mean()
            avg_output_tkns =  df['output_tkns'].mean()

            results[name] = {
                "Success Rate (%)": round(success_pct, 2),
                "Avg Time (s)": round(avg_time, 4),
                'Avg Prompt tkns': int(round(avg_prompt_tkns)),
                'Avg Output tkns': int(round(avg_output_tkns))
            }

    # Convert results to a readable DataFrame
    summary_df = pd.DataFrame(results).T
    return summary_df

# Usage:
for csv_file_datapath in csv_files:
    data = pd.read_csv(csv_file_datapath)
    print(csv_file_datapath, f'total examples: {data.shape[0]}', f"Average success rate: {round((data['iverilog_output'] == 'OK').mean() * 100 , 2)}%" )
    print(analyze_dataset(data))
    print('---------------------------------------------------')

sv_results_yi-coder_9b_3.csv total examples: 200 Average success rate: 77.5%
                     Success Rate (%)  Avg Time (s)  Avg Prompt tkns  \
combinational_basic             82.50        4.4468            574.0   
sequential_basic                92.50       11.1624            842.0   
fsm                             43.33       37.1658           2841.0   
industry                        20.00       89.9768           9647.0   

                     Avg Output tkns  
combinational_basic            186.0  
sequential_basic               465.0  
fsm                           1514.0  
industry                      3808.0  
---------------------------------------------------
sv_results_codegemma_7b_3.csv total examples: 200 Average success rate: 85.0%
                     Success Rate (%)  Avg Time (s)  Avg Prompt tkns  \
combinational_basic              87.5        4.5666            493.0   
sequential_basic                 85.0       12.2921            722.0   
fsm                  

In [4]:
import pandas as pd
import glob

# 1. Grab all .csv files in the current directory
# Adjust the path if your files are in a specific folder
files = glob.glob("*.csv")

all_failed_rows = []

for file in files:
    # Skip the output file if it already exists to avoid recursion
    if file == "pass2fortgts.csv":
        continue

    df = pd.read_csv(file)

    # 2. Filter for rows where iverilog_output is NOT 'OK'
    # This handles both different capitalization and leading/trailing spaces
    failed_df = df[df['iverilog_output'].astype(str).str.strip().str.upper() != 'OK']

    # 3. Keep only the requested columns
    subset = failed_df[['original_code', 'generated_code', 'iverilog_output']]

    all_failed_rows.append(subset)

# 4. Combine everything and export
if all_failed_rows:
    final_df = pd.concat(all_failed_rows, ignore_index=True)
    final_df.to_csv('pass2fortgts.csv', index=False)
    print(f"Successfully created pass2fortgts.csv with {len(final_df)} rows.")
else:
    print("No matching rows found.")

Successfully created pass2fortgts.csv with 199 rows.
